In [1]:
import pyarrow.parquet as pq



In [2]:
table = pq.read_table("test_parsing_parquet_pyarrow")

table.schema

page_id: int32
type: string
coord: list<element: int32>
  child 0, element: int32
content: string
description: string
id: int32
file_name: string
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1049

In [5]:
rows = table.to_pylist()

len(rows)

203

In [15]:
rows[9]

{'page_id': 3,
 'type': 'section_header',
 'coord': [194, 196, 522, 241],
 'content': '1 Introduction',
 'description': None,
 'id': 9,
 'file_name': 'OpenAI-Privacy-Filter-Model-Card.pdf'}

In [34]:
import json
import re
import pyarrow.parquet as pq
from typing import Any, Dict, List


def read_parquet_rows(path: str) -> List[Dict[str, Any]]:
    table = pq.read_table(path)
    return table.to_pylist()


def is_real_section_header(content: str) -> bool:
    """
    True only for real numbered section headers, e.g.
    '7.5.3 One-Hop Reasoning Evals'

    False for:
    'Table 5: ...'
    'Figure 2: ...'
    """

    content = (content or "").strip()

    if content.lower().startswith(("table ", "figure ")):
        return False

    return bool(
        re.match(r"^\d+(\.\d+)*\s+[A-Z].+", content)
    )


def build_unit_id(
    section: str,
    page_ids: List[int],
    source_ids: List[int],
) -> str:
    section_slug = re.sub(
        r"[^a-zA-Z0-9]+",
        "_",
        section.lower(),
    ).strip("_")

    pages = "_".join(map(str, sorted(set(page_ids))))
    sources = "_".join(map(str, source_ids))

    return f"{section_slug}_p{pages}_{sources}"


def build_section_units(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    rows = sorted(
        rows,
        key=lambda r: (
            r.get("page_id", 0),
            r.get("id", 0),
        ),
    )

    sections = []
    current_section = None

    for row in rows:
        row_type = row.get("type")
        content = row.get("content") or ""

        if row_type == "section_header" and is_real_section_header(content):
            if current_section is not None:
                sections.append(current_section)

            current_section = {
                "section_name": content.strip(),
                "text_parts": [],
                "table_parts": [],
                "page_ids": [row.get("page_id")],
                "source_ids": [row.get("id")],
                "file_name": row.get("file_name"),
            }
            continue

        if current_section is not None:
            if row_type == "text":
                current_section["text_parts"].append(content)

            elif row_type == "table":
                current_section["table_parts"].append(content)

            elif row_type == "section_header" and content.lower().startswith("table "):
                current_section["table_parts"].append(content)

            elif row_type == "section_header" and content.lower().startswith("figure "):
                current_section["text_parts"].append(content)

            else:
                continue

            current_section["page_ids"].append(row.get("page_id"))
            current_section["source_ids"].append(row.get("id"))

    if current_section is not None:
        sections.append(current_section)

    units = []

    for s in sections:
        units.append({
            "unit_id": build_unit_id(
                section=s["section_name"],
                page_ids=s["page_ids"],
                source_ids=s["source_ids"],
            ),
            "type": "mixed",
            "section": s["section_name"],
            "text": "\n\n".join(s["text_parts"]),
            "table": "\n\n".join(s["table_parts"]),
            "page_ids": sorted(set(s["page_ids"])),
            "source_ids": s["source_ids"],
            "file_name": s["file_name"],
        })

    return units

In [42]:
rows = read_parquet_rows("test_parsing_parquet_pyarrow")

text_blocks = build_section_units(rows)

print(len(text_blocks))
print(text_blocks[33])

36
{'unit_id': '7_5_3_one_hop_reasoning_evals_p15_16_166_167_168_170_171_172_173_174_175', 'type': 'mixed', 'section': '7.5.3 One-Hop Reasoning Evals', 'text': 'We also evaluate Privacy Filter on examples that require a single step of contextual reasoning to determine that a span should be treated as PII. In these cases, the sensitive value is not explicitly labeled at the point where it appears. Instead, the model must connect it to an earlier statement that defines an alias or reference, such as a phrase indicating that a later token sequence corresponds to an account number or national identification number.\n\nRepresentative examples include prompts of the form: “For verification purposes, when I say ‘marigold’ later on, I’m referring to my residential electric utility account number; I’ll provide it at the very end,” followed much later by “As mentioned earlier, ‘marigold’ is 7281-0543-98217.”\n\nSimilarly, another example defines “starlight” as a government-issued national identi

In [43]:
import re
from typing import Any, Dict, List, Optional

from openai import OpenAI


OPENAI_MODEL = "gpt-4.1-mini"
ENABLE_LLM_SUMMARIES = True

_llm_client: Optional[OpenAI] = None


USER_PROMPT = """Analyze the following text. Based on the provided instructions, identify the main themes of the text and return the result in the following format (strictly as a single string):
- A text string containing a neutral and concise summary of the provided content. The summary must be composed of very short, informative sentences, separated by semicolons.
- The separator symbol $$$
- If the text does not contain understandable or relevant information, return: Other ** brief description
"""


SYSTEM_PROMPT = """You are an AI assistant specialized in the analysis of texts. Your task is to analyze the text.

# Objective and format
Your task is to identify the main themes of the text and return a neutral and concise summary in the following format:
<Theme1> ** <brief description1>$$$<Theme2> ** <brief description2>

# General rules
1. Analyze the text.

2. The summary must be composed of a few very short, informative, and neutral sentences. Each micro-sentence must follow this structure:
   - Theme ** brief description

3. Use the symbol ** to separate the theme name from its description, and the symbol $$$ to separate multiple themes.

4. Ignore emojis, repeated characters, excessive punctuation, uppercase/lowercase differences, and non-essential temporal or geographical details.

5. If the text does not contain elements that belong to any theme, return exactly:
   Other ** brief description
   - Example: "Other ** legislative references"

6. If the text is not understandable or is irrelevant, return exactly:
   Other ** Unclear or Irrelevant
"""



In [55]:
def _get_llm_client() -> OpenAI:
    global _llm_client
    if _llm_client is None:
        _llm_client = OpenAI()  # uses OPENAI_API_KEY env var
    return _llm_client


def build_text_for_enrichment(b: Dict[str, Any]) -> str:
    """
    Costruisce il testo completo da dare al modello per una section unit.
    Include:
    - titolo sezione
    - testo
    - tabelle
    """

    parts = []

    if b.get("section"):
        parts.append(f"SECTION:\n{b['section']}")

    if b.get("text"):
        parts.append(f"TEXT:\n{b['text']}")

    if b.get("table"):
        parts.append(f"TABLES:\n{b['table']}")

    return "\n\n".join(parts).strip()


def ai_act_topics_openai(text: str, model: str = OPENAI_MODEL) -> str:
    """Returns ONLY a string in the format: Tema ** desc$$$Tema2 ** desc2 or Altro ** ..."""
    client = _get_llm_client()
    resp = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT + "\n\nTESTO:\n" + (text or "")},
        ],
    )
    out = (resp.output_text or "").strip()
    if not out:
        return "Altro ** Incomprensibile o Irrilevante"
    out = re.sub(r"\s*\n+\s*", " ", out).strip()
    return out

def parse_themes_string(s: str) -> List[Dict[str, str]]:
    """
    Converte: "Tema1 ** desc1$$$Tema2 ** desc2"
    in: [{"tema":"Tema1","descrizione":"desc1"}, {"tema":"Tema2","descrizione":"desc2"}]
    """
    s = (s or "").strip()
    if not s:
        return [{"tema": "Altro", "descrizione": "Incomprensibile o Irrilevante"}]

    parts = [p.strip() for p in s.split("$$$") if p.strip()]
    out: List[Dict[str, str]] = []

    for p in parts:
        if "**" not in p:
            out.append({"tema": "Other", "description": p.strip()})
            continue
        tema, desc = p.split("**", 1)
        tema = tema.strip() or "Other"
        desc = desc.strip() or "description"
        out.append({"theme": tema, "description": desc})

    return out if out else [{"tema": "Altro", "descrizione": "Unclear or Irrelevant"}]


def enrich_text_blocks_with_summaries(
    text_blocks: List[Dict[str, Any]],
    model: str = OPENAI_MODEL,
) -> List[Dict[str, Any]]:
    """
    Per ogni text_block prodotto da build_section_units:
      - costruisce il contenuto della sezione
      - fa 1 chiamata OpenAI per section unit
      - parse in lista themes[]
      - aggiunge raw_summary, summary, themes, model
    """

    enriched: List[Dict[str, Any]] = []

    for b in text_blocks:
        block_text = build_text_for_enrichment(b)

        if not block_text:
            raw_summary = None
            themes = []
            used_model = None
        elif ENABLE_LLM_SUMMARIES:
            raw_summary = ai_act_topics_openai(block_text, model=model)
            themes = parse_themes_string(raw_summary)
            used_model = model
        else:
            raw_summary = None
            themes = []
            used_model = None

        b2 = dict(b)
        b2["semantic_annotations"] = themes


        enriched.append(b2)

    return enriched

In [68]:
# Costruzione section units
text_blocks = build_section_units(rows)

# Prendiamo solo una unità
sample_block = [text_blocks[1]]

print(sample_block[0]["section"])
print(sample_block[0]["text"][:1000])
print(sample_block[0]["table"][:1000])


2 Model Details
Privacy Filter is a bidirectional token classification model with span decoding. It is trained in
phases, beginning with autoregressive pretraining. The pretrained language model is then modified
and post-trained as a bidirectional banded attention token classifier with band size 128 (effective
attention window: 257 tokens including self). At inference time, we apply constrained sequence
decoding to produce coherent BIOES (Begin, Inside, Outside, End, Single) span labels.



In [48]:
sample_block

[{'unit_id': '2_model_details_p4_21_22',
  'type': 'mixed',
  'section': '2 Model Details',
  'text': 'Privacy Filter is a bidirectional token classification model with span decoding. It is trained in\nphases, beginning with autoregressive pretraining. The pretrained language model is then modified\nand post-trained as a bidirectional banded attention token classifier with band size 128 (effective\nattention window: 257 tokens including self). At inference time, we apply constrained sequence\ndecoding to produce coherent BIOES (Begin, Inside, Outside, End, Single) span labels.',
  'table': '',
  'page_ids': [4],
  'source_ids': [21, 22],
  'file_name': 'OpenAI-Privacy-Filter-Model-Card.pdf'}]

In [56]:

# Enrichment SOLO di quella
enriched_sample = enrich_text_blocks_with_summaries(
    sample_block,
    model=OPENAI_MODEL,
)

print("\nTHEMES:\n")
for t in enriched_sample[0]["semantic_annotations"]:
    print(t)


THEMES:

{'theme': 'Model Architecture', 'description': 'Description of the Privacy Filter as a bidirectional token classification model with span decoding; Training Process ** Explanation of phased training starting with autoregressive pretraining and followed by post-training as a bidirectional banded attention token classifier; Inference Method ** Use of constrained sequence decoding to generate coherent BIOES span labels'}


In [57]:
enriched_sample

[{'unit_id': '2_model_details_p4_21_22',
  'type': 'mixed',
  'section': '2 Model Details',
  'text': 'Privacy Filter is a bidirectional token classification model with span decoding. It is trained in\nphases, beginning with autoregressive pretraining. The pretrained language model is then modified\nand post-trained as a bidirectional banded attention token classifier with band size 128 (effective\nattention window: 257 tokens including self). At inference time, we apply constrained sequence\ndecoding to produce coherent BIOES (Begin, Inside, Outside, End, Single) span labels.',
  'table': '',
  'page_ids': [4],
  'source_ids': [21, 22],
  'file_name': 'OpenAI-Privacy-Filter-Model-Card.pdf',
  'semantic_annotations': [{'theme': 'Model Architecture',
    'description': 'Description of the Privacy Filter as a bidirectional token classification model with span decoding; Training Process ** Explanation of phased training starting with autoregressive pretraining and followed by post-trainin

In [69]:
# 1. Arricchisci tutte le unità
enriched_units = enrich_text_blocks_with_summaries(
    text_blocks,
    model=OPENAI_MODEL,
)

print(f"Enriched units: {len(enriched_units)}")

Enriched units: 36


In [70]:
from datetime import datetime
from neo4j import GraphDatabase
import os
import dotenv


load_status = dotenv.load_dotenv()
print("Load status:", load_status)

if load_status is False:
    raise RuntimeError("Environment variables not loaded.")

URI = os.getenv("NEO4J_URI")

AUTH = (
    os.getenv("NEO4J_USERNAME"),
    os.getenv("NEO4J_PASSWORD"),
)
neo4j_driver = GraphDatabase.driver(URI, auth=AUTH)
neo4j_driver.verify_connectivity()
print("Connection established.")


def save_enriched_units_to_graph(
    document_id: str,
    document_title: str,
    enriched_units: list[dict],
) -> dict:
    timestamp = datetime.utcnow().isoformat()

    units = []

    for u in enriched_units:
        clean_annotations = []

        for ann in u.get("semantic_annotations", []):
            theme_name = ann.get("theme") or ann.get("tema") or ann.get("name")
            theme_desc = ann.get("description") or ann.get("descrizione") or ""

            if not theme_name:
                continue

            clean_annotations.append({
                "theme": str(theme_name),
                "description": str(theme_desc),
            })

        units.append({
            "unit_id": u["unit_id"],
            "unit_type": u.get("type", ""),
            "section": u.get("section", ""),
            "summary": u.get("summary", ""),
            "text_preview": u.get("text", "")[:500],
            "table_preview": u.get("table", "")[:500],
            "page_ids": u.get("page_ids", []),
            "source_ids": u.get("source_ids", []),
            "file_name": u.get("file_name", ""),
            "model": u.get("model", ""),
            "theme_names": [a["theme"] for a in clean_annotations],
            "semantic_annotations": clean_annotations,
        })

    cypher = """
    MERGE (d:Document {document_id: $document_id})
    SET d.title = $document_title,
        d.updated_at = $timestamp

    WITH d
    UNWIND $units AS unit

    MERGE (u:ContextUnit {unit_id: unit.unit_id})
    SET u.display_name = unit.section,
        u.section = unit.section,
        u.unit_type = unit.unit_type,
        u.summary = unit.summary,
        u.text_preview = unit.text_preview,
        u.table_preview = unit.table_preview,
        u.page_ids = unit.page_ids,
        u.source_ids = unit.source_ids,
        u.file_name = unit.file_name,
        u.model = unit.model,
        u.theme_names = unit.theme_names,
        u.updated_at = $timestamp

    MERGE (d)-[:HAS_UNIT]->(u)

    WITH u, unit
    UNWIND unit.semantic_annotations AS ann

    MERGE (t:Theme {name: ann.theme})
    SET t.description = ann.description,
        t.updated_at = $timestamp

    MERGE (u)-[r:HAS_THEME]->(t)
    SET r.description = ann.description,
        r.updated_at = $timestamp
    """

    with neo4j_driver.session() as session:
        session.run(
            cypher,
            document_id=document_id,
            document_title=document_title,
            timestamp=timestamp,
            units=units,
        )

    return {
        "status": "saved",
        "document_id": document_id,
        "saved_units": len(units),
    }

Load status: True
Connection established.


In [71]:
result = save_enriched_units_to_graph(
    document_id="openai_privacy_filter_model_card",
    document_title="OpenAI Privacy Filter Model Card",
    enriched_units=enriched_units,
)

C:\Users\mvall\AppData\Local\Temp\ipykernel_29664\2912497808.py:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat()
